# Model 3: Helmet Detection on Motorcycles
**YOLOv8 training on Kaggle GPU**

Detects whether the motorcycle rider is wearing a helmet.
Classes: `helmet`, `no_helmet`

In [ ]:
!pip install ultralytics -q

In [ ]:
import os, yaml, shutil
from ultralytics import YOLO

# ── Configuration ──────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/your-helmet-dataset'   # <-- CHANGE THIS
PROJECT_NAME = 'helmet_detection'
MODEL_BASE   = 'yolov8n.pt'
EPOCHS       = 60
IMG_SIZE     = 640
BATCH_SIZE   = 16
# ───────────────────────────────────────────────────────────────

In [ ]:
data_yaml = os.path.join(DATASET_PATH, 'data.yaml')

if not os.path.exists(data_yaml):
    config = {
        'path': DATASET_PATH,
        'train': 'train/images',
        'val':   'val/images',
        'nc': 2,
        'names': ['helmet', 'no_helmet']
    }
    data_yaml = '/kaggle/working/helmet_data.yaml'
    with open(data_yaml, 'w') as f:
        yaml.dump(config, f)
    print('Created data.yaml')

with open(data_yaml) as f:
    print(f.read())

In [ ]:
model = YOLO(MODEL_BASE)

results = model.train(
    data    = data_yaml,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = '/kaggle/working/runs',
    name    = PROJECT_NAME,
    device  = 0,
    patience= 20,
    save    = True,
    plots   = True,
    # Helmet detection: head is small — increase scale sensitivity
    scale   = 0.5,
    translate = 0.1,
    degrees = 10.0,
    fliplr  = 0.5,
    mosaic  = 0.7,
    copy_paste = 0.2,
    close_mosaic = 10,
)

In [ ]:
best_weights = f'/kaggle/working/runs/{PROJECT_NAME}/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=data_yaml, imgsz=IMG_SIZE)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
output_path = '/kaggle/working/model3_helmet.pt'
shutil.copy(best_weights, output_path)
print(f'Model saved: {output_path}')